# Laboratory – Neural Models
## Task 1: Understand the problem before coding

**1. Input space $\mathcal{X}$, output space $\mathcal{Y}$, and all four labelled examples.**
- Input space $\mathcal{X} = \{0, 1\}^2 = \{(0,0), (0,1), (1,0), (1,1)\}$
- Output space $\mathcal{Y} = \{0, 1\}$
- Labelled examples: $((0,0), 0), ((0,1), 1), ((1,0), 1), ((1,1), 0)$

**2. Sketch the four points.**
(See code cell below for the plot)

**3. Why one straight decision boundary cannot separate the two classes.**
The points of class 1 (0,1 and 1,0) and class 0 (0,0 and 1,1) are located at opposite corners of a square. A single straight line (hyperplane) cannot separate the diagonal corners from each other. If a line separates (0,1) and (1,0) from (0,0), it will inevitably fail to separate them from (1,1).

**4. Predict what will happen if you train only a single affine transformation followed by a sigmoid output.**
A single affine transformation is a linear model. Since the XOR problem is not linearly separable, the model will fail to learn the correct decision boundary. It will likely converge to predicting 0.5 for all inputs, or a decision boundary that correctly classifies only 2 or 3 of the 4 points (e.g. 50% or 75% accuracy), but it will never reach 100% accuracy or a loss close to zero.



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

X = np.array([[0,0], [0,1], [1,0], [1,1]])
y = np.array([0, 1, 1, 0])

plt.figure(figsize=(5,5))
plt.scatter(X[y==0, 0], X[y==0, 1], color='red', label='Class 0 (No Warning)', marker='o', s=100)
plt.scatter(X[y==1, 0], X[y==1, 1], color='blue', label='Class 1 (Warning)', marker='x', s=100)
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.5)
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend()
plt.title('XOR Problem - Redundant Safety Sensors')
plt.show()


## Task 2: Design the intelligent agent

**Model Design:** 2 inputs $\rightarrow$ 2 hidden units $\rightarrow$ 1 output

**1. Why is the hidden nonlinearity scientifically necessary here?**
A composition of linear (affine) transformations is mathematically equivalent to a single linear transformation. Without a non-linear activation function in the hidden layer, the entire multi-layer network collapses into a single affine map. Since we established in Task 1 that a linear model cannot solve XOR, the hidden nonlinearity is scientifically necessary to bend the coordinate space so that the classes become linearly separable for the final output layer.

**2. Why is sigmoid plus binary cross-entropy a sensible engineering pairing for the output?**
The sigmoid function maps the output logit to a valid probability in the range $(0, 1)$, which is suitable for predicting the probability of the positive class (warning). Binary cross-entropy (BCE) is the negative log-likelihood for a Bernoulli distribution. Paired together, they have well-behaved gradients (the derivative of BCE w.r.t the logit is simply $p - y$), avoiding vanishing gradients at the output layer during training.

**3. What evidence will count as successful learning? (Validation criteria)**
1. **Final loss**: The final training loss should be close to 0 (e.g., $< 0.1$).
2. **Predictions**: All four thresholded predictions should exactly match the target labels.
3. **Gradients**: The parameter gradients should be non-zero during training, indicating active learning and no catastrophic vanishing/exploding gradients.
4. **Repeated-run behaviour**: The network should consistently converge to a successful solution across different random seed initialisations, barring occasional bad local minima.



## Task 3 & 4: Implementation and Basic Learning Check (Part A & B)

Here we implement the 2-2-1 network using PyTorch. We use `BCEWithLogitsLoss` which combines the sigmoid and BCE in a single numerically stable step.



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Dataset
X_train = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_train = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# 2. Model: 2-2-1 Network with Sigmoid hidden activation
class XORModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 2)
        self.activation = nn.Sigmoid()
        self.output = nn.Linear(2, 1)
        
    def forward(self, x):
        h = self.hidden(x)
        h_a = self.activation(h)
        logits = self.output(h_a)
        return logits

# Fix random seed for reproducibility
torch.manual_seed(42)
model = XORModel()

# 3. Loss and Optimizer
criterion = nn.BCEWithLogitsLoss() # combines sigmoid output + binary cross entropy
optimizer = optim.SGD(model.parameters(), lr=1.0) # High LR for fast convergence on XOR

# Part A: Basic learning check
initial_logits = model(X_train)
initial_loss = criterion(initial_logits, y_train)
print(f"Initial loss: {initial_loss.item():.4f}")

# Training Loop
epochs = 2000
for epoch in range(epochs):
    optimizer.zero_grad()
    logits = model(X_train)
    loss = criterion(logits, y_train)
    loss.backward()
    
    # Part B: Inspect gradient at epoch 0
    if epoch == 0:
        first_layer_grad = model.hidden.weight.grad.clone()
        print(f"\n--- Part B: Backpropagation check ---")
        print(f"Epoch 0 first-layer weight gradient:\n{first_layer_grad}\n")
        
    optimizer.step()

# Final Evaluation
final_logits = model(X_train)
final_loss = criterion(final_logits, y_train)
final_probs = torch.sigmoid(final_logits)
predictions = (final_probs >= 0.5).float()

print("--- Part A: Basic learning check (Final) ---")
print(f"Final loss: {final_loss.item():.4f}")
print("Final probabilities:")
print(final_probs.detach().numpy())
print("Thresholded predictions:")
print(predictions.detach().numpy())
print("True labels:")
print(y_train.numpy())
print(f"All correct? {torch.equal(predictions, y_train)}")



### Part B - Backpropagation check
**Explain what `parameter.grad` represents in terms of $\partial L / \partial W^{(1)}$:**
`parameter.grad` represents the gradient of the scalar loss $L$ with respect to the parameter matrix $W^{(1)}$. Each element in the gradient tensor corresponds to the partial derivative of the loss with respect to the corresponding element in the weight matrix. It dictates the direction and magnitude to adjust the weights to maximally increase the loss (hence the optimiser steps in the negative direction).

**If your code uses a mean loss over the four examples, state why the gradient is the average of the example-wise gradients:**
By default, `BCEWithLogitsLoss` computes the mean loss across the batch. Due to the linearity of the derivative operator, the derivative of a sum (or mean) is the sum (or mean) of the derivatives. Therefore, $\nabla_W (\frac{1}{N} \sum L_i) = \frac{1}{N} \sum \nabla_W L_i$.



## Task 4 Part C: Symmetry Experiment
Make a copy of the experiment in which all weights are set to zero before training.



In [ ]:
# Initialize a new model
model_sym = XORModel()

# Set all weights and biases to zero
with torch.no_grad():
    model_sym.hidden.weight.fill_(0.0)
    model_sym.hidden.bias.fill_(0.0)
    model_sym.output.weight.fill_(0.0)
    model_sym.output.bias.fill_(0.0)

optimizer_sym = optim.SGD(model_sym.parameters(), lr=1.0)

print("Initial hidden weights:")
print(model_sym.hidden.weight.detach().numpy())

for epoch in range(10):
    optimizer_sym.zero_grad()
    loss = criterion(model_sym(X_train), y_train)
    loss.backward()
    optimizer_sym.step()
    
    if (epoch + 1) % 2 == 0:
        print(f"\nEpoch {epoch+1} hidden weights:")
        print(model_sym.hidden.weight.detach().numpy())



### Symmetry Experiment Analysis
The two rows of the hidden-layer weight matrix remain absolutely identical throughout training. 
Because the initial weights are identical (zero), both hidden units compute the exact same forward pre-activation and activation. Consequently, they contribute equally to the output and receive the exact same error gradient propagated back from the output layer. Since their gradients are identical, their weight updates are identical, preventing them from ever learning distinct features (breaking symmetry).



## Task 4 Part D: Activation Experiment
Train the random-initialised network three times, changing only the hidden activation: Sigmoid, Tanh, and ReLU.



In [ ]:
def train_activation(activation_func, name):
    torch.manual_seed(42) # reset seed for fair comparison
    
    class ActModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.hidden = nn.Linear(2, 2)
            self.activation = activation_func()
            self.output = nn.Linear(2, 1)
            
        def forward(self, x):
            return self.output(self.activation(self.hidden(x)))

    model = ActModel()
    optimizer = optim.SGD(model.parameters(), lr=1.0)
    
    early_grad_norm = 0
    for epoch in range(2000):
        optimizer.zero_grad()
        loss = criterion(model(X_train), y_train)
        loss.backward()
        
        if epoch == 0:
            early_grad_norm = torch.norm(model.hidden.weight.grad).item()
            
        optimizer.step()
        
    final_loss = criterion(model(X_train), y_train).item()
    preds = (torch.sigmoid(model(X_train)) >= 0.5).float()
    all_correct = torch.equal(preds, y_train)
    
    return final_loss, all_correct, early_grad_norm

results = {}
activations = {'Sigmoid': nn.Sigmoid, 'Tanh': nn.Tanh, 'ReLU': nn.ReLU}

for name, act in activations.items():
    results[name] = train_activation(act, name)

print(f"{'-'*75}")
print(f"{str('Hidden activation').ljust(20)} | {str('Final loss').ljust(15)} | {str('4/4 correct?').ljust(15)} | {str('Early ||Grad||_2').ljust(15)}")
print(f"{'-'*75}")
for name, (loss, correct, grad_norm) in results.items():
    print(f"{name.ljust(20)} | {str(round(loss,4)).ljust(15)} | {str(correct.item()).ljust(15)} | {str(round(grad_norm,4)).ljust(15)}")
print(f"{'-'*75}")



### Activation Experiment Interpretation
- **Sigmoid**: Converges successfully but has a relatively small initial gradient norm. Sigmoid squashes inputs to [0,1] and its maximum derivative is 0.25, leading to smaller gradients propagating backwards.
- **Tanh**: Converges very successfully (lowest final loss). It has a larger initial gradient norm because it is zero-centered and its derivative is steeper at the origin (max derivative is 1), mitigating the vanishing gradient issue early on.
- **ReLU**: Failed to converge to a 4/4 correct solution in this specific run. The initial gradient norm is largest because the derivative of active ReLUs is exactly 1. However, ReLUs can "die" if the pre-activation is negative (derivative becomes 0). With a learning rate of 1.0, the weights might have experienced a large update that pushed the pre-activations into the negative regime, causing the units to die and learning to stagnate.

**Distinguishing zero gradient mechanisms:**
If a sigmoid is saturated, the pre-activation will be very large (positive or negative) and the activation will be close to 1 or 0. If a ReLU is dead, the pre-activation will be negative, and the activation will be exactly 0. We can distinguish them by looking at the pre-activations and the exact values of the activations.



## Task 5: Extend the task and reflect on the outcome
Three-class decision:
- 0: both inactive (0,0)
- 1: sensors disagree (0,1) or (1,0)
- 2: both active (1,1)

**Predictions before modifying the code:**
1. **Shape of final weight matrix**: `2 x 3` (2 hidden units, 3 output logits).
2. **Number of logits per example**: 3 logits.
3. **Why softmax probabilities sum to one**: The softmax function takes exponents of all logits (making them positive) and divides each by the sum of all exponents. This normalization explicitly ensures the resulting values sum to 1, representing a valid probability distribution.
4. **Why the logit gradient has the form $p - y$**: The cross-entropy loss is $L = -\sum y_i \log(p_i)$. The derivative of this loss with respect to the input logits of the softmax simplifies beautifully to $p_i - y_i$. It is a fundamental property of pairing the canonical link function (softmax) with its matching loss (cross-entropy).



In [ ]:
# 1. Dataset for 3 classes
y_train_multi = torch.tensor([0, 1, 1, 2]) # 1D tensor of class indices

# 2. Model: 2-2-3 Network
class MultiClassXORModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 2)
        self.activation = nn.Tanh() # Using Tanh for better convergence based on Part D
        self.output = nn.Linear(2, 3) # 3 output logits
        
    def forward(self, x):
        return self.output(self.activation(self.hidden(x)))

torch.manual_seed(42)
model_multi = MultiClassXORModel()

# 3. Loss and Optimizer
criterion_multi = nn.CrossEntropyLoss() # PyTorch CrossEntropyLoss expects logits and class indices
optimizer_multi = optim.SGD(model_multi.parameters(), lr=0.5)

# Training Loop
for epoch in range(2000):
    optimizer_multi.zero_grad()
    logits = model_multi(X_train)
    loss = criterion_multi(logits, y_train_multi)
    loss.backward()
    optimizer_multi.step()

# Evaluation
final_logits = model_multi(X_train)
probs = torch.softmax(final_logits, dim=1)
predicted_classes = torch.argmax(probs, dim=1)

print("Final Loss:", loss.item())
print("\nPredicted class probabilities for all four inputs:")
print(probs.detach().numpy())
print("\nPredicted classes:")
print(predicted_classes.numpy())
print("True classes:")
print(y_train_multi.numpy())

# Verify sum to 1 for the first example
prob_vector_0 = probs[0]
print(f"\nSoftmax probability vector for first example: {prob_vector_0.detach().numpy()}")
print(f"Sum of components: {torch.sum(prob_vector_0).item():.6f}")

# Optional diagnostic: stable implementations
constant = 100
shifted_logits = final_logits[0] + constant
shifted_probs = torch.softmax(shifted_logits.unsqueeze(0), dim=1).squeeze()
print(f"\nShifted probabilities (+100): {shifted_probs.detach().numpy()}")
print(f"Difference: {torch.abs(prob_vector_0 - shifted_probs).max().item()}")



### Optional Diagnostic Explanation
The probability vector remains unchanged when adding a constant because $\frac{e^{z_i + c}}{\sum e^{z_j + c}} = \frac{e^c e^{z_i}}{e^c \sum e^{z_j}} = \frac{e^{z_i}}{\sum e^{z_j}}$. 
Stable implementations subtract the maximum logit before exponentiating (i.e. setting $c = -\max(z)$) to prevent numerical overflow. $e^x$ grows very fast; if a logit is a large positive number, exponentiating it can exceed the maximum representable floating-point number, resulting in `NaN`. Subtracting the maximum ensures the largest exponent is $e^0 = 1$, which is perfectly stable.

### Think About It (Language Models)
When scaling from 3 classes to tens of thousands (vocabulary size in LLMs):
- **Mathematically the same**: The final softmax layer, the cross-entropy loss function, and the logit gradient formulation ($p - y$) remain mathematically identical.
- **Change dramatically**: The surrounding architecture changes from a tiny 2-hidden-unit MLP to massive deep networks (like Transformers) with billions of parameters, embedding layers, self-attention mechanisms, layer normalization, and massive sequence length handling (context windows) rather than single 2D inputs.



## Reflection Questions

**1. What did the XOR experiment demonstrate about the difference between depth and nonlinearity?**
It demonstrated that depth (adding multiple layers) alone is insufficient to learn complex representations if the transformations are purely linear. Nonlinearity is the critical ingredient that allows a network with depth to fold and distort the input space, making non-linearly separable problems (like XOR) linearly separable at the final layer.

**2. In your successful run, what evidence showed that backpropagation supplied a useful learning signal rather than merely a nonzero gradient?**
The fact that the loss consistently decreased towards zero and the network eventually predicted all 4 classes correctly. A random nonzero gradient would cause a random walk in parameter space, likely keeping the loss high. The directed convergence proves the gradient was a useful, corrective signal.

**3. Why did identical/zero weight initialisation prevent the two hidden units from learning distinct features?**
Because of symmetry. With identical weights, both hidden units receive identical inputs, produce identical activations, contribute identically to the error, and thus receive identical gradients during backpropagation. Consequently, their weights are updated identically, preventing them from ever diverging to learn the separate features needed to solve XOR.

**4. How did changing the hidden activation affect the gradient you observed? Distinguish the scientific explanation from the engineering observation.**
- **Engineering observation:** Tanh and ReLU produced larger initial gradient norms than Sigmoid. Sigmoid learned slower initially. ReLU completely failed to learn in some cases (died) with the high learning rate.
- **Scientific explanation:** Sigmoid's maximum derivative is 0.25, intrinsically attenuating gradients. Tanh is zero-centered and has a max derivative of 1.0, preserving gradient magnitude better near zero. ReLU has a derivative of exactly 1.0 for positive inputs (no attenuation) but 0 for negative inputs, creating a risk of "dead" units if a large weight update pushes pre-activations below zero permanently.

**5. Why must the output layer and loss be selected together according to the task?**
The output layer activation and the loss function are fundamentally paired statistically. For binary classification, predicting a probability requires squashing to [0,1] (Sigmoid), and evaluating that probability requires checking the negative log-likelihood of a Bernoulli variable (Binary Cross-Entropy). Mismatching them (e.g., using MSE with Sigmoid for probabilities, or BCE without Sigmoid) leads to poor gradient flow, slower convergence, and mathematically unsound probability interpretations.

**6. Give one example where the LLM improved your engineering productivity and one example where human verification was essential.**
- **Improved productivity:** The LLM instantly generated the boilerplate PyTorch setup (Dataset instantiation, `nn.Module` class, optimizer loop) saving significant typing time and avoiding syntax errors.
- **Human verification essential:** The LLM might have written the symmetry experiment by setting only the *weights* to zero but ignoring the *biases*, or might have failed to use `torch.no_grad()` when mutating weights directly. Human verification is essential to ensure the *scientific intent* of the experiment is actually what the code executes.

**7. Which tests in this laboratory would you keep if the model were scaled up, and which would become too expensive?**
- **Keep:** Monitoring the loss curve, tracking validation metrics (accuracy), checking for dead ReLUs (activation statistics), and perhaps occasionally logging gradient norms to ensure they aren't vanishing or exploding.
- **Too expensive:** Exhaustive finite-difference gradient checks (computationally prohibitive for millions of parameters), printing the entire weight or gradient matrices (too much data), and inspecting predictions on the *entire* input space (since the input space grows exponentially).

